In [0]:
import csv
from collections import Counter

RUTA_FUENTE = "/Volumes/workspace/taco_bueno/archivos_fuente/"

archivos = [
    archivo
    for archivo in dbutils.fs.ls(RUTA_FUENTE)
    if archivo.name.lower().endswith(".csv")
]

inventario = []

for archivo in archivos:
    try:
        ruta_local = archivo.path.removeprefix("dbfs:")

        with open(
            ruta_local,
            "r",
            encoding="utf-8-sig",
            newline=""
        ) as contenido:
            lector = csv.reader(contenido)

            encabezado = next(lector, None)

            primera_fila = next(
                (
                    fila
                    for fila in lector
                    if any(valor.strip() for valor in fila)
                ),
                None
            )

        if not encabezado:
            estado = "Archivo vacío"
        elif primera_fila is None:
            estado = "Solo encabezado"
        else:
            estado = "Con datos"

        inventario.append((
            archivo.name,
            len(encabezado) if encabezado else 0,
            estado,
            encabezado or [],
            None
        ))

    except Exception as error:
        inventario.append((
            archivo.name,
            None,
            "Error de lectura",
            [],
            str(error)
        ))

inventario_df = spark.createDataFrame(
    inventario,
    schema="""
        archivo STRING,
        columnas INT,
        estado STRING,
        encabezado ARRAY<STRING>,
        error STRING
    """
)

display(inventario_df.orderBy("archivo"))

print("Resumen:")
for estado, cantidad in Counter(
    fila[2] for fila in inventario
).items():
    print(f"{estado}: {cantidad}")

print("\nEncabezados distintos:")
encabezados = {
    tuple(fila[3])
    for fila in inventario
    if fila[3]
}

for encabezado in sorted(encabezados):
    print(f"\n{len(encabezado)} columnas:")
    print(list(encabezado))